# Biopython으로 코로나바이러스 서열 분석해보기
## 개요
전 세계를 동결시켰던 바이러스... 그죠. 코로나바이러스죠. 저는 자가격리도 해보고 걸려도 봤습니다. 자가격리는 초창기에 직장 동료가 확진돼서 했던거고... 그때도 뭐 다들 '헐 답답하겠다' 했는데, 저는 본투비 집순이라 크게 데미지는 없었습니다. 평소에도 포켓몬고 하는거나 심부름(아주 드물게 볼일) 아니면 잘 안나가서 부모님이 제발 좀 나가라고 하실 정도었으니까요. 

걸린건 오미크론때였는데, 처음에 저는 이게 좀 크게 아픈 감기인 줄 알았습니다. 근데 검사해보니까 어머나 세상에. 확진이야. 그때는 일 다닐때라 재택근무 했었죠. 

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, muscle
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns

# Biopython
from Bio import Entrez, SeqIO # 왼쪽: 일단 털어보자/오른쪽: 시퀀스 다루려면 필요합니다. 필수임. 
from Bio import AlignIO # 서열 분석해줄 친구
from Bio import Phylo # 트리 그릴라면 필요해요 
from Bio.Align import AlignInfo
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio.Seq import Seq

import io # 누구세요?
import subprocess # 서브 프로세스(이건 또 뭐여...)
from collections import defaultdict
from collections import Counter

# 통계분석용
from scipy.stats import mannwhitneyu
from itertools import combinations
from scipy.stats import spearmanr

# 군집분석용
import kmedoids
from sklearn.metrics import silhouette_score

# 주성분분석
from sklearn.decomposition import PCA

In [ ]:
# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumsquare_ac' # 나눔바른펜(본인 기본 고딕 싫어함)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 사전세팅
Entrez.email = "blackholekun@gmail.com" # 이메일 
muscle_exe = "/opt/homebrew/bin/muscle" # 이거 경로 있어야 써요(which 치면 나옴)

# NCBI의 창고는 얼웨이즈 오픈입니다. 

In [ ]:
# 쿼리 조건: SARS-CoV-2 (코로나19), Spike 단백질 위주로 털어보기
# 2025년 최신 데이터 + 사람 숙주 조건
query = "SARS-CoV-2 AND S[Gene Name] AND 2025[PDAT] AND Homo sapiens[Host]"

# 1. ID 리스트 가져오기
handle = Entrez.esearch(db="nucleotide", term=query, retmax=300)
record = Entrez.read(handle)
id_list = record["IdList"]
handle.close()

# 2. 실제 서열 데이터 가져오기 (FASTA 형식)
fetch_handle = Entrez.efetch(db="nucleotide", id=id_list, rettype="fasta", retmode="text")
sequences = list(SeqIO.parse(fetch_handle, "fasta"))
fetch_handle.close()

# 3. 저장 
with open("influenza_h3n2.fasta", "w") as f:
    SeqIO.write(sequences, f, "fasta")

print(f"성공적으로 {len(sequences)}개의 서열을 가져왔습니다.")
print("----------")

for record in sequences[:3]:
    print(f"ID: {record.id}")
    print(f"Description: {record.description}")
    print(f"Length: {len(record.seq)} bp\n")

- 아니 근데 왜 스파이크만 털어요? 쟤 게놈이 일단 길고요... 항원항체반응이랑 관련 있는게 스파이크입니다. 

In [ ]:
# 콤퓨타에 저-장
vir_sequence = []
for i, id in enumerate(id_list):
    print(f"Downloading sequence {i+1}/{len(id_list)}: {id}")
    handle = Entrez.efetch(db="nucleotide", id=id, rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    
    # record에 id와 seq가 다 들어가야되더라... (안되면 오류남 봤음)
    vir_sequence.append(record) 

# 파일로 저장
SeqIO.write(vir_sequence, "hantavirus_sequence.fasta", "fasta")
print('Done!')

In [ ]:
# 시퀀스 길이 체크 
for rec in vir_sequence:
    print(f"ID: {rec.id} | Length: {len(rec.seq)}")

- 아이고... 시퀀스가 닥터 스트레인지라도 만났나... 대혼돈의 유니버스입니다. 여기서 스파이크만 거르는 작업이 필요해요. 

## 스파이크만 거르기

In [ ]:
# 가져온 sequences 리스트에서 길이로 필터링
spike_only = [rec for rec in sequences if 3000 <= len(rec.seq) <= 4500]

# 필터링된 결과 저장
with open("sars_cov2_spike_clean.fasta", "w") as f:
    SeqIO.write(spike_only, f, "fasta")

print(f"전체 {len(sequences)}개 중 진짜 스파이크만 {len(spike_only)}개 골라냈습니다.")

# MSA
- 가라! 머슬! 

In [ ]:
print('MSA start... ')

# MSA 분석 시-작
try: 
    result = subprocess.run([muscle_exe, "-align", "sars_cov2_spike_clean.fasta", "-output", "sars_cov2_spike_align.fasta"], check=True, capture_output=True, text=True)
    print("Completed. ")
except subprocess.CalledProcessError as e: 
    print(f"MSA failed: {e}")
finally:
    alignment = AlignIO.read("influenza_h3n2_muscle_aligned.fasta", "fasta")

# 밥 먹고 오면 끝나있겠는데...? 

In [ ]:
print("====== MSA Result ======")
alignment = AlignIO.read("sars_cov2_spike_align.fasta", "fasta") # FASTA 니네 확장자가 몇개냐... 

for record in alignment:
    print(f"{record.id[:10]:<15} : {record.seq[:100]}")

## 평균 보존율

In [ ]:
def calculate_conservation_no_gap(alignment, gap_threshold=0.5):
    length = alignment.get_alignment_length()
    scores = []

    for i in range(length):
        column_raw = alignment[:, i]

        # gap 비율이 너무 높으면 제외 (선택사항)
        gap_fraction = column_raw.count("-") / len(column_raw)
        if gap_fraction > gap_threshold:
            continue

        # gap 제거
        column = column_raw.replace("-", "")
        if len(column) == 0:
            continue

        # 최빈 염기 비율 = 보존도
        most_common = max(set(column), key=column.count)
        score = column.count(most_common) / len(column)
        scores.append(score)

    return scores

scores = calculate_conservation_no_gap(alignment)

print(f"해당 구간의 평균 보존율: {np.mean(scores)*100:.2f}%")

- 어... 이게... 맞아...? 너 바이러스 아냐...? 

# 섀넌 엔트로피

In [ ]:
def calculate_shannon_entropy(alignment, gap_threshold=0.5):
    length = alignment.get_alignment_length()
    entropies = []

    for i in range(length):
        column_raw = alignment[:, i]
        
        # gap 비율 계산
        gap_fraction = column_raw.count("-") / len(column_raw)
        if gap_fraction > gap_threshold:
            continue  # gap 많은 position 제거
        
        # gap 제거
        column = column_raw.replace("-", "")
        if len(column) == 0:
            continue
        
        counts = Counter(column)
        total = sum(counts.values())
        
        entropy = 0
        for c in counts.values():
            p = c / total
            entropy -= p * np.log2(p)
        
        entropies.append(entropy)

    return entropies

def sliding_window_mean(values, window=20):
    """
    values : np.array (entropy scores, np.nan 포함)
    window : window size
    """
    smoothed = []

    for i in range(len(values)):
        start = max(0, i - window // 2)
        end = min(len(values), i + window // 2 + 1)

        window_vals = values[start:end]
        window_vals = window_vals[~np.isnan(window_vals)]

        if len(window_vals) == 0:
            smoothed.append(np.nan)
        else:
            smoothed.append(np.mean(window_vals))

    return np.array(smoothed)

In [ ]:
entropies = calculate_shannon_entropy(alignment)

H_max = np.log2(4)
variation_scores = [e / H_max for e in entropies]  # 0~1 스케일

In [ ]:
plt.figure(figsize=(15, 7))
plt.plot(variation_scores, alpha=0.8, color='#00498c')
plt.fill_between(range(len(variation_scores)), variation_scores, alpha=0.3)

plt.axvline(1322, linestyle='--', color='red', alpha=0.5)
plt.text(1322, max(variation_scores)*1.07, 'Spike POS 1322', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.axvline(3295, linestyle='--', color='red', alpha=0.5)
plt.text(3295, max(variation_scores)*1.07, 'Spike POS 3295', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.axvline(626, linestyle='--', color='red', alpha=0.5)
plt.text(626, max(variation_scores)*1.07, 'Spike POS 626', color='red', alpha=0.5, ha='center', fontweight='bold')

plt.title("Viral Variation Hotspots", y = 1.07)
plt.xlabel("Alignment Position (filtered)")
plt.ylabel("Normalized Shannon Entropy")
plt.show()

## 변이 핫스팟

In [ ]:
# 섀넌 엔트로피 점수 도출
def get_top_variable_sites_no_gap(alignment, top_n=10):
    length = alignment.get_alignment_length()
    variability = []

    ref_seq = alignment[0].seq

    for i in range(length):
        # 🔴 reference가 gap이면 무조건 스킵
        if ref_seq[i] == '-':
            continue

        column = alignment[:, i].replace("-", "")
        if not column:
            continue

        counts = Counter(column)
        total = sum(counts.values())

        entropy = 0.0
        for c in counts.values():
            p = c / total
            entropy -= p * math.log2(p)

        variability.append((i, entropy))

    return sorted(variability, key=lambda x: x[1], reverse=True)[:top_n]

def alignment_to_sequence_pos(aligned_seq, aln_pos):
    count = 0
    for i in range(aln_pos + 1):
        if aligned_seq[i] != '-':
            count += 1
    return count


ref_seq = alignment[0].seq
top_sites = get_top_variable_sites_no_gap(alignment, top_n=10)

high_entropy_ha_sites = []

print("--- 변이가 집중된 주요 포지션 분석 결과 ---")
for aln_pos, score in top_sites:
    real_pos = alignment_to_sequence_pos(ref_seq, aln_pos)
    high_entropy_ha_sites.append(real_pos)
    print(f"Alignment {aln_pos:4d} → Spike Pos {real_pos:4d} | 엔트로피: {score:.3f}")

print("\n최종 고엔트로피 포지션 리스트:")
print(high_entropy_ha_sites)

## 맨 휘트니 U 검정
- DNA가 염기입니다 여러분. 수치형이 아니예요. 평균이고 분산이고 몰라요 걔는. 

### 거 통계값좀 내봅시다. 

In [ ]:
entropy_raw = np.array(entropies)

mean_raw = np.mean(entropy_raw)
median_raw = np.median(entropy_raw)
iqr_raw = np.percentile(entropy_raw, 75) - np.percentile(entropy_raw, 25)

print("[Raw entropy]")
print(f"Mean:   {mean_raw:.4f}")
print(f"Median: {median_raw:.4f}")
print(f"IQR:    {iqr_raw:.4f}")

In [ ]:
plt.hist(entropy_raw, bins=50, color="#00498c")
plt.title("Raw Shannon Entropy Distribution (Site-wise)")
plt.xlabel("Entropy (bits)")
plt.ylabel("Frequency")
plt.show()

- 아니 저게 맞아요?

In [ ]:
# Windowed entrophy
entropy_window = sliding_window_mean(entropy_raw, window=25)
entropy_window_valid = entropy_window[~np.isnan(entropy_window)]

mean_win = np.mean(entropy_window_valid)
median_win = np.median(entropy_window_valid)
iqr_win = (
    np.percentile(entropy_window_valid, 75)
    - np.percentile(entropy_window_valid, 25)
)

print("[Windowed entropy]")
print(f"Mean:   {mean_win:.4f}")
print(f"Median: {median_win:.4f}")
print(f"IQR:    {iqr_win:.4f}")

In [ ]:
plt.hist(entropy_window_valid, bins=50, color="#00498c")
plt.title("Windowed Shannon Entropy Distribution (Regional)")
plt.xlabel("Mean entropy (windowed)")
plt.ylabel("Frequency")
plt.show()

- 이게 진짜 맞다고? 

### 가설
- 귀무가설: 코로나바이러스 스파이크의 변이는 무작위적으로 발생하며, 특정 위치에 선호적으로 집중되지 않는다.
- 대립가설: 코로나바이러스 스파이크의 변이는 무작위가 아니며, 특정 위치(hotspots)에 유의하게 집중된다.

In [ ]:
# --- 2. normalization ---
entropy_min = np.nanmin(entropy_window)
entropy_max = np.nanmax(entropy_window)

entropy_window_normalized = (
    entropy_window - entropy_min
) / (entropy_max - entropy_min)

variation_scores = entropy_window_normalized

# --- 3. NaN 제거 (🔥 중요) ---
valid_scores = variation_scores[~np.isnan(variation_scores)]

# --- 4. hotspot threshold (normalized 기준) ---
threshold_norm = np.percentile(valid_scores, 90)
threshold_raw = threshold_norm * (entropy_max - entropy_min) + entropy_min

hotspots = valid_scores[valid_scores >= threshold_norm]
non_hotspots = valid_scores[valid_scores < threshold_norm]

u_stat, p_value = mannwhitneyu(
    hotspots,
    non_hotspots,
    alternative="greater"
)

print(f"Hotspot threshold (top 10%, normalized entropy): {threshold_norm:.3f}")
print(f"Corresponding raw entropy threshold: {threshold_raw:.3f}")
print(f"Mann–Whitney U statistic: {u_stat:.1f}")
print(f"p-value: {p_value:.4e}" if p_value > 1e-10 else "p-value: <1e-10")

- <속보> 귀무가설 압도적 기각 
> 코로나바이러스 스파이크의 변이는 무작위가 아니며, 변이가 자주 터지는 특정 포인트가 있다. 

# Phylogenic tree

In [ ]:
# 트! 리! 
calculator = DistanceCalculator('identity')
dm = calculator.get_distance(alignment)
constructor = DistanceTreeConstructor(calculator, 'nj')
tree = constructor.build_tree(alignment)

terms = tree.get_terminals()
x_limit = max([tree.distance(t) for t in terms])
fig = plt.figure(figsize=(16, 18), dpi=150) # 난 해상도 설정도 될 줄 몰랐고... 
ax = fig.add_subplot(1, 1, 1)

for clade in tree.get_terminals():
    original_name = str(clade.name)
    if '_' in original_name:
        parts = original_name.split('_')
        clade.name = f"[{parts[0]}] {parts[1]} ({original_name})"
    else:
        clade.name = original_name

Phylo.draw(tree, axes=ax, do_show=False, label_func=lambda x: "", show_confidence=False)

# 내가 진짜 이것때문에 제미나이랑 급나 씨름했는데 색깔이 안바껴요. 
# 이름도 몇번이나 했는데 ID만 줄창떠요. 아오. 
for i, node in enumerate(terms):
    y_pos = i + 1  # 가지의 y축 위치
    x_pos = tree.distance(node) # 가지가 끝나는 x축 위치
    
    orig_name = str(node.name)
    # 이름 가공: [연도] 지역 (ID)
    if '_' in orig_name:
        p = orig_name.split('_')
        # 혹시 이미 가공된 이름이라면 중복 방지
        display_text = f"  ◀ [{p[0]}] {p[1]}" if '[' not in orig_name else f"  ◀ {orig_name}"
    else:
        display_text = f"  ◀ {orig_name}"
    
    # 가지 끝(x_pos)에 바로 텍스트를 박습니다.
    ax.text(x_pos, y_pos, display_text, 
            va='center', ha='left', 
            fontsize=14, 
            fontweight='bold' if "LC909067" in orig_name else 'normal')

ax.set_xlim(0, x_limit * 1.8) 
ax.set_ylim(0, len(terms) + 2)
ax.set_axis_off() # 축 숫자 빠잉 

plt.rc('font', size=14) # 내부 글꼴 사이즈
plt.rc('axes', titlesize=20) # 제모옥은 이 크기로 하겠습니다 
plt.title("Coronavirus spike Phylogenetic Tree by Region/Year")
plt.tight_layout()
plt.savefig("Coronavirus_Final_Tree.png", dpi=300, bbox_inches='tight')
plt.xlabel("Genetic Distance (Substitutions per site)")
plt.show()

## 군집분석

In [ ]:
# Biopython 거리 행렬 → numpy array 변환
labels = dm.names
dist_array = np.array([[dm[i, j] for j in labels] for i in labels])

# 실루엣 점수로 최적 K 찾기
silhouette_scores = []
K_range = range(2, 10)

for k in K_range:
    dist_matrix = pairwise_distances(dist_array, metric='precomputed') if False else dist_array
    km = kmedoids.KMedoids(n_clusters=k, method='fasterpam', random_state=42)
    km.fit(dist_array)
    score = silhouette_score(dist_array, km.labels_, metric='precomputed')
    silhouette_scores.append(score)

plt.figure(figsize=(12, 9))
plt.plot(K_range, silhouette_scores, marker='o', color='#5f4b8b')
plt.title('실루엣 점수로 최적 K 찾기')
plt.xlabel('K')
plt.ylabel('Silhouette Score')
plt.show()

In [ ]:
km = kmedoids.KMedoids(n_clusters=2, method='fasterpam', random_state=42)
km.fit(dist_array)

labels = np.array(km.labels_).astype(int)
medoid_indices = km.medoid_indices_

print("=== 2개 군집 대표 서열 ===")
for i, idx in enumerate(medoid_indices):
    rec = alignment[int(idx)]
    original = next((r for r in alignment if r.id.split('.')[0] == rec.id.split('.')[0]), None)
    if original:
        print(f"군집 {i}: {original.description}")
    else:
        print(f"군집 {i}: {rec.id}")

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(dist_array)

plt.figure(figsize=(12, 9))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=km.labels_, palette='Set2')
plt.title('K-Medoids 군집 시각화 (K=2)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

- 저기 혼자 떨어져 있는 애가 계통수에서도 혼자 놀던 애입니다. 

## 통계분석-스피어맨 상관계수
- 아... 여러분... 이거 인플루엔자때 봤다 그죠? 이게 뭐냐면 스피어맨 상관계수입니다. 네. 
- 근데 왜 맨휘트니 안하고 이걸 하냐고요? 위에서부터 쭉 보셨겠지만 일단 300개 가져와서 그중에서 스파이크만 걸러서 26개 나왔는데, 걔네들 엔트로피 계산해보니까 변이 구역이 지이이이이이이이이이인짜 미미합니다. 맨휘트니가 안돼요. 
- 이유는 인플루엔자랑 비슷합니다. 근데 얘는 한 아종에서 갈라진게 아니라 포인트가 너무 적어요. 

### 가설
- 귀무가설: 트리 거리와 서열 유사도는 상관이 없다 → 계통수 구조는 서열 차이를 반영하지 않는다.
- 대립가설: 트리 거리와 서열 유사도간에 서로 상관이 있다 → 계통수 구조는 서열 차이를 반영했다. 

In [ ]:
tree_distances = []
seq_identities = []

terms = tree.get_terminals()

def pairwise_identity(seq1, seq2):
    # 두 서열 중 어느 한쪽이라도 갭이 아닌 위치만 골라냄
    matches = 0
    total_valid_length = 0
    for s1, s2 in zip(seq1, seq2):
        if s1 == '-' and s2 == '-': # 둘 다 갭이면 무시
            continue
        total_valid_length += 1
        if s1 == s2:
            matches += 1
    
    return (matches / total_valid_length) if total_valid_length > 0 else 0

for rec1, rec2 in combinations(alignment, 2):
    id1 = rec1.id.split('.')[0]
    id2 = rec2.id.split('.')[0]
    
    try:
        # 가공된 트리 이름 속에서 원본 ID가 포함된 노드를 각각 찾음
        node1 = [t for t in terms if id1 in t.name][0]
        node2 = [t for t in terms if id2 in t.name][0]
        
        d = tree.distance(node1, node2)
        iden = pairwise_identity(str(rec1.seq), str(rec2.seq))
        
        tree_distances.append(d)
        seq_identities.append(iden)
    except IndexError:
        # 트리에 해당 ID가 없는 경우 건너뜀
        continue

In [ ]:
rho, p = spearmanr(tree_distances, seq_identities)

print(f'Rho: {rho:.4f}')
print(f'P-value: {p:.4e}')

### 시각화

In [ ]:
plt.figure(figsize=(10, 7), dpi=120)
sns.regplot(x=tree_distances, y=seq_identities, 
            scatter_kws={'alpha':0.2, 'color':'gray', 's':10}, 
            line_kws={'color':'red', 'label': f'Spearman Rho: {rho:.3f}'})

plt.title("Coronavirus: Tree Distance vs Sequence Identity", fontsize=15, pad=15)
plt.xlabel("Genetic Distance on Tree", fontsize=12)
plt.ylabel("Pairwise Sequence Identity", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

- 쟤는 게놈을 일부 거르고 해서 먼지(?)가 좀 덜한겁니다. 네. 
- 인플루엔자때랑 마찬가지로 귀무가설은 압도적으로 기각당했군요. 
> 트리 거리와 서열 유사도간에 서로 상관이 있다 → 계통수 구조는 서열 차이를 반영했다. 